# Double Zernike Focal-Plane Fitting

Fits focal-plane Noll Zernike polynomials to per-image donut wavefront
residuals using robust regression (Huber M-estimator). Runs both z1toz3
(piston+tilt+tip) and z1toz6 (adds defocus+astigmatism) fits.

**Input**: Donut parquet table from `intrinsics_mktable.ipynb`  
**Output**: Fit parameter table (parquet) with per-image coefficients, errors, and quality flags

In [ ]:
from pathlib import Path

# ============================================================
# Parameters
# ============================================================

# Coordinate system: 'CCS' or 'OCS'
coord_sys = 'OCS'

# Input HDF5 file (from intrinsics_mktable, contains donuts + visits)
prefix = 'fam_danish'
wep_ver = 'wep_v16_8_0'
dviz_ver = 'dviz_v3_5_0'
day_obs_min = 20260315
day_obs_max = 20260317

output_dir = 'output'
input_file = f'{output_dir}/{prefix}_{wep_ver}_{dviz_ver}_{day_obs_min}_{day_obs_max}.hdf5'

# Output fit table (default: {input_stem}_fits.parquet)
output_file = f'{input_file.rsplit(".", 1)[0]}_fits.parquet'

# Bad fit thresholds
bad_fit_threshold = 2.0  # μm
min_donuts = 200

print(f'Input:  {input_file}')
print(f'Output: {output_file}')

In [ ]:
from code.dz_fitting import run_double_zernike_fits

fit_merged = run_double_zernike_fits(
    input_file=input_file,
    coord_sys=coord_sys,
    output_file=output_file,
    bad_fit_threshold=bad_fit_threshold,
    min_donuts=min_donuts,
)

In [ ]:
import numpy as np

# Display summary
print(f'Fit table: {len(fit_merged)} images')
print(f'Columns: {fit_merged.colnames}')
print(f'Bad fits: {np.sum(fit_merged["bad_fit"])}')
print()

# Show sample z1toz3 Z4 coefficients
if 'z1toz3_z4_c1' in fit_merged.colnames:
    for ci in range(1, 4):
        col = f'z1toz3_z4_c{ci}'
        vals = np.array(fit_merged[col])
        print(f'{col}: [{vals.min():.3f}, {vals.max():.3f}] μm, median={np.median(vals):.3f}')

print()
# Show first few rows
fit_merged[['day_obs', 'seq_num', 'n_donuts', 'bad_fit']][:10]